# linear_panel_fe

> 对应代码：`EconometricsCode/code_in_notes/Chap.14/linear_panel_fe.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.14`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.14")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

先清空环境并读入俄罗斯 NTV 电视覆盖与各选区选举得票的面板数据 NTV_Aggregate_Data_reshaped.dta。

In [ ]:
%%stata
clear
set more off
use ../datasets/NTV_Aggregate_Data_reshaped.dta

声明面板结构：截面单位为大区（tik_id），时间维度为 year。

In [ ]:
%%stata
xtset tik_id year

构造处理变量 Watch_probit_p：1995 年 NTV 尚未成立，因此该年所有地区的观看概率统一设为 0；1999 年及以后取实际估计出的观看概率 Watch_probit。这样处理变量在 1995 年无变异、1999 年存在截面差异，从而可以在面板中识别 NTV 覆盖对选举结果的影响。

In [ ]:
%%stata
// 1995年NTV并没有成立
gen Watch_probit_p=0
replace Watch_probit_p=Watch_probit if year==1999

用 reghdfe 做 LSDV 回归：absorb(i.tik_id) 吸收大区固定效应，等价于为每个大区放入虚拟变量，控制所有不随时间变化的大区特征；i.year 加入年份固定效应控制共同时间冲击；cluster(region) 在 region 层面聚类标准误。if year!=2003 剔除 2003 年样本。

In [ ]:
%%stata
// LSDV回归
reghdfe Votes_SPS_ Watch_probit_p i.year if year!=2003, absorb(i.tik_id) cluster(region)

用 Stata 自带的 xtreg, fe 重新估计同一个固定效应模型，fe 选项表示组内（within）估计量，结果应与 reghdfe 完全一致，这里用作对照验证。

In [ ]:
%%stata
// xtreg 回归
xtreg Votes_SPS_ Watch_probit_p i.year if year!=2003, fe cluster(region)

改用一阶差分估计：对得票率与处理变量分别取 4 期差分，然后用 OLS 回归差分后的方程，并保留年份虚拟变量与 region 层面聚类标准误。在两期或近似两期的面板中，一阶差分与固定效应估计量数值上等价，都可以消除不随时间变化的个体异质性，读者可比较此处系数与前面两个命令的结果。

In [ ]:
%%stata
// 使用一阶差分
gen delta_Votes_SPS_=Votes_SPS_-L4.Votes_SPS_
gen delta_Watch_probit_p=Watch_probit_p-L4.Watch_probit_p
reg delta_Votes_SPS_ delta_Watch_probit_p i.year if year!=2003, cluster(region)